In [1]:
import os
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torch.distributed as dist
import torch.multiprocessing as mp
import sys

sys.path.append("./") # replace "./" as your project path
sys.path.append("./utils")
os.chdir("./")

from  utils import *
from builder import *
from dataloader import *
from densenet import *
from embedding_extractor_test import *
from preprocessing import *
from utils import *
from parse import *
import densenet as models

from builder import *
from parse import *

In [2]:
class MoCoDataset_test(torch.utils.data.Dataset):
    def __init__(self, filepath, transform = None):
        self.path = filepath
        self.transform = transform

    def __len__(self):
        self.data = np.load(self.path,allow_pickle=True)
        return self.data['x'].shape[0]   
        
    def __getitem__(self,i):
        
        x = self.data['x'][i]
        label = self.data['y'][i]
        return x, label


def load_para(save_path):
    checkpoint = torch.load(save_path)
    #载入模型参数
    state_dict = checkpoint['state_dict']
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k[7:]  # remove 'module.' of dataparallel
        if name.endswith(".weight") or name.endswith(".bias"):
            if name.split(".")[2] in ["0","2"]:
                name = f'{name.split(".")[0]}.{name.split(".")[1]}.{name.split(".")[3]}'
        new_state_dict[name] = v
    return new_state_dict



/mnt/hpc/home/shilei/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
args = moco_parser().parse_args('')
#args
args = moco_parser().parse_args('')
model_dict = models.__dict__

kwargs = {"randomzero" : 0.3}
model = MoCo(
        model_dict["densenet27"],
        num_batches = 32, 
        in_features = args.in_features, 
        dim=args.moco_dim, 
        K=args.moco_k, 
        m=args.moco_m, 
        T=args.moco_t, 
        mlp=args.mlp,
        **kwargs)

In [4]:
#载入数据

test_dataset = MoCoDataset_test("./data/zscore_test_Depmap_ProteinCodingGene_mRNA.npz")
train_dataset = MoCoDataset_test("./data/zscore_train_Depmap_ProteinCodingGene_mRNA.npz")

In [5]:
#载入checkpoint文件
#save_path = "/mnt/hpc/home/shilei/RongfangNie/project/Nierongfang/TumorDrugPredict/code/moco-main copy 2/model/checkpoint_0041.pth copy.tar"
save_path = "//result/mRNA_protein/checkpoint_0199.pth.tar"
model.load_state_dict(load_para(save_path))
model.eval()

/tmp/ipykernel_26428/2462996228.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(save_path)


MoCo(
  (encoder_q): DenseNet(
    (drop1): Dropout(p=0.3, inplace=False)
    (features): Sequential(
      (conv0): Linear(in_features=10000, out_features=7168, bias=False)
      (norm0): GroupNorm(1, 7168, eps=1e-05, affine=True)
      (relu0): ReLU(inplace=True)
      (denseblock1): _DenseBlock(
        (denselayer1): _DenseLayer(
          (norm1): GroupNorm(1, 7168, eps=1e-05, affine=True)
          (relu1): ReLU(inplace=True)
          (conv1): Linear(in_features=7168, out_features=4096, bias=False)
          (norm2): GroupNorm(1, 4096, eps=1e-05, affine=True)
          (relu2): ReLU(inplace=True)
          (conv2): Linear(in_features=4096, out_features=1024, bias=False)
        )
        (denselayer2): _DenseLayer(
          (norm1): GroupNorm(1, 8192, eps=1e-05, affine=True)
          (relu1): ReLU(inplace=True)
          (conv1): Linear(in_features=8192, out_features=4096, bias=False)
          (norm2): GroupNorm(1, 4096, eps=1e-05, affine=True)
          (relu2): ReLU(inplace

In [6]:
test_dataloader = DataLoader(test_dataset,  batch_size=128)
train_dataloader = DataLoader(train_dataset, batch_size=128)

In [7]:
def model_out(train_dataloader,model):
    X = []
    Y = []
    for batch in train_dataloader:
        data,label= batch
        q = torch.tensor(data,dtype=torch.float32)
        #q = q.unsqueeze(0)
        #frature = q.detach().numpy()[0]
        frature = model.encoder_q(q).detach().numpy()
        X = X + frature.tolist()
        Y = Y + list(label)
    X = np.array(X)
    Y = np.array(Y)
    return X,Y

In [8]:
#测试集
X,Y = model_out(test_dataloader,model)

/tmp/ipykernel_26428/1596742536.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  q = torch.tensor(data,dtype=torch.float32)


In [9]:
X1, Y1 = model_out(train_dataloader,model)

/tmp/ipykernel_26428/1596742536.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  q = torch.tensor(data,dtype=torch.float32)


In [10]:
feature_Data = pd.DataFrame(data = X, index=Y)
feature_Data1 = pd.DataFrame(data = X1, index=Y1)

In [11]:
Out_Data = pd.concat([feature_Data,feature_Data1])

In [13]:
out_dir = "../data/DrugComb/Process"

Out_Data.to_pickle(out_dir + "/MOCO_feature_Depmap_ProteinCodingGene.pkl")